In [2]:
from datetime import datetime
import pandas as pd
import numpy as np
import sklearn.model_selection

# Load the data
df = pd.read_csv("../data/citibike_weather_daily_clean.csv")

In [ ]:
# M1: Split the Data
from sklearn.model_selection import train_test_split

# Load the clean CSV
df = pd.read_csv("../data/citibike_weather_daily_clean.csv")

print("Dataset shape:", df.shape)
print("\nColumns available:", list(df.columns))

# Check for missing values
print("\nMissing values before handling:")
print(df.isna().sum()[df.isna().sum() > 0])

# Handle missing values: drop rows with NaN in feature columns
# (This respects the "Rules of the Road" - we need clean data for modeling)
df_clean = df.dropna(subset=['temp_f', 'precip_in', 'wind_speed_knots'])

print(f"\nRows before dropping NaN: {len(df)}")
print(f"Rows after dropping NaN: {len(df_clean)}")
print(f"Rows removed: {len(df) - len(df_clean)}")

# Define features (exclude ride_date and target variable)
# Core features: temperature, precipitation, wind, day-of-week dummies, trend
feature_cols = [
    'temp_f',           # Temperature feature
    'precip_in',        # Precipitation
    'wind_speed_knots', # Wind
    'day_Monday',       # Day-of-week dummies
    'day_Tuesday',
    'day_Wednesday',
    'day_Thursday',
    'day_Friday',
    'day_Saturday',
    'day_Sunday',
    'days_since_launch' # Trend feature
]

# Create feature matrix X and target y
X = df_clean[feature_cols]
y = df_clean['num_rides']

# Split into training and test sets (80/20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42  # Set for reproducibility
)

print(f"\nTraining set size: {X_train.shape[0]} rows")
print(f"Test set size: {X_test.shape[0]} rows")
print(f"Features used: {len(feature_cols)}")
print(f"\nFeature list:\n{feature_cols}")

# Store df_clean for later use in residual plots
df = df_clean

In [5]:
# M2: Fit the Core Model
from sklearn.linear_model import LinearRegression

# Create and fit the linear regression model
model = LinearRegression()
model.fit(X_train, y_train)

print("✓ Linear Regression model fitted successfully!")
print(f"\nModel intercept: {model.intercept_:.2f}")
print(f"Number of coefficients: {len(model.coef_)}")

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

In [ ]:
# M3: Evaluate
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Make predictions on both training and test sets
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Calculate R² scores
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

# Calculate error metrics on test set (in real units: rides)
mae_test = mean_absolute_error(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

print("="*60)
print("MODEL EVALUATION")
print("="*60)
print(f"\nR² Scores:")
print(f"  Training R²: {r2_train:.4f}")
print(f"  Test R²:     {r2_test:.4f}")

print(f"\nTest Set Error Metrics (in rides):")
print(f"  MAE (Mean Absolute Error):  {mae_test:.2f} rides")
print(f"  RMSE (Root Mean Squared Error): {rmse_test:.2f} rides")

print(f"\nDifference between RMSE and MAE: {rmse_test - mae_test:.2f} rides")
print(f"Ratio (RMSE/MAE): {rmse_test/mae_test:.2f}")

## M3: Evaluation Interpretation

### What does the test R² mean?
The test R² tells us the proportion of variance in daily ridership that our model can explain using weather, day of week, and the growth trend. For example, if R² = 0.75, it means our model explains 75% of the variation in ride counts, and 25% remains unexplained (due to factors we didn't measure or random variation).

### Why do we care about TEST R² more than TRAINING R²?
- **Training R²** shows how well the model fits data it has already seen - it can be artificially high due to overfitting
- **Test R²** shows how well the model generalizes to NEW, unseen data - this is what matters in the real world
- If training R² >> test R², the model has memorized patterns that don't generalize (overfitting)
- Test R² is the honest measure of predictive performance

### MAE Translation for Operations Team
**"Our prediction is typically off by [MAE value] rides."**

This means on an average day, if we predict X rides, the actual count will be within ±[MAE] of that prediction.

### RMSE vs MAE Comparison
RMSE is always ≥ MAE because it squares errors before averaging (penalizing large errors more heavily). A large gap between RMSE and MAE indicates that a few days have very large prediction errors.

**Which days are likely badly-missed?**
- Days with extreme or unusual weather (heat waves, major storms)
- Holidays or special events (July 4th, parades, etc.)
- Days with unusual combinations of conditions the model hasn't learned well

In [ ]:
# M4: Interpret the Coefficients

# Create a dataframe of feature names and their coefficients
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print("="*70)
print("MODEL COEFFICIENTS (sorted by absolute value)")
print("="*70)
print(coef_df.to_string(index=False))

print("\n" + "="*70)
print("TOP 3 MOST INTERESTING COEFFICIENTS - PLAIN ENGLISH TRANSLATION")
print("="*70)

# Get top 3 by absolute value
top_3 = coef_df.head(3)

for idx, row in top_3.iterrows():
    feature = row['Feature']
    coef = row['Coefficient']
    
    print(f"\n{feature}: {coef:.2f}")
    
    # Translate to plain English
    if 'precip' in feature:
        print(f"  → Each additional inch of rain costs us roughly {abs(coef):.0f} rides.")
    elif 'temp' in feature:
        if coef > 0:
            print(f"  → Each additional degree (F) adds roughly {coef:.0f} rides.")
        else:
            print(f"  → Each additional degree (F) costs us roughly {abs(coef):.0f} rides.")
    elif 'wind' in feature:
        print(f"  → Each additional knot of wind costs us roughly {abs(coef):.0f} rides.")
    elif 'day_' in feature:
        day = feature.replace('day_', '')
        if coef > 0:
            print(f"  → {day}s see roughly {coef:.0f} MORE rides than the baseline.")
        else:
            print(f"  → {day}s see roughly {abs(coef):.0f} FEWER rides than the baseline.")
    elif 'days_since_launch' in feature:
        print(f"  → Each day after launch adds roughly {coef:.2f} rides (growth trend).")
        annual_growth = coef * 365
        print(f"     (That's roughly {annual_growth:.0f} rides per year of growth)")

print("\n" + "="*70)
print("Do the signs match EDA expectations?")
print("="*70)
print("Review the coefficient signs and compare to your EDA findings.")
print("Flag any surprises - e.g., unexpected positive/negative relationships.")

In [ ]:
# M5: Diagnose With Residuals
import matplotlib.pyplot as plt

# Calculate residuals (actual - predicted)
residuals_test = y_test - y_test_pred

# Get the test set indices to align with ride_date
test_dates = df.loc[y_test.index, 'ride_date']

# Convert ride_date to datetime if it's not already
test_dates = pd.to_datetime(test_dates)

# Create two residual plots
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Residuals vs Predicted Values
axes[0].scatter(y_test_pred, residuals_test, alpha=0.5)
axes[0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0].set_xlabel('Predicted Rides')
axes[0].set_ylabel('Residuals (Actual - Predicted)')
axes[0].set_title('Residuals vs Predicted Values')
axes[0].grid(True, alpha=0.3)

# Plot 2: Residuals vs Date
axes[1].scatter(test_dates, residuals_test, alpha=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Residuals (Actual - Predicted)')
axes[1].set_title('Residuals vs Date')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print statistics about residuals
print("="*60)
print("RESIDUAL ANALYSIS")
print("="*60)
print(f"Mean of residuals: {residuals_test.mean():.2f} (should be ~0)")
print(f"Std dev of residuals: {residuals_test.std():.2f}")
print(f"Min residual: {residuals_test.min():.2f}")
print(f"Max residual: {residuals_test.max():.2f}")

# Find the worst predictions
worst_predictions = pd.DataFrame({
    'date': test_dates,
    'actual': y_test,
    'predicted': y_test_pred,
    'residual': residuals_test
}).sort_values('residual', key=abs, ascending=False).head(10)

print("\n10 Worst Predictions:")
print(worst_predictions.to_string(index=False))

## M5: Residual Diagnosis

### Patterns to Look For:

**In the Residuals vs Predicted plot:**
- **Random scatter around 0** = good! The model has no systematic bias
- **Funnel shape** (wider at high predictions) = variance increases with ridership
- **Curve pattern** = non-linear relationship we're missing (might need polynomial features)
- **Clumps** = distinct groups the model treats differently

**In the Residuals vs Date plot:**
- **Random scatter** = good! No time-based patterns missed
- **Upward/downward drift** = trend feature isn't capturing growth properly
- **Seasonal waves** = missing seasonal patterns (could add month dummies or sine/cosine transforms)
- **Clusters of high errors** = specific time periods with unusual behavior

### What the Model is Telling Us:
A residual plot with structure is the model's way of showing what features it's missing. Look for:
- Systematic under/over-prediction at certain times
- Patterns that suggest non-linear relationships
- Seasonal or cyclical patterns not captured by current features

In [ ]:
# M6: Improve One Thing
# Based on residual diagnosis, make ONE improvement

# Example improvement: Add a quadratic temperature feature
# (Temperature relationship may not be linear - EDA showed peak around 60-70°F)

print("="*70)
print("MODEL IMPROVEMENT EXPERIMENT")
print("="*70)
print("\nChange being tested: Adding temperature squared (temp_f²)")
print("Rationale: EDA showed ridership peaks at moderate temps, suggesting")
print("           a non-linear (quadratic) relationship with temperature.")
print("="*70)

# Create new feature set with temperature squared
X_improved = X.copy()
X_improved['temp_f_squared'] = X['temp_f'] ** 2

# Split the improved feature set
X_train_imp, X_test_imp, y_train_imp, y_test_imp = train_test_split(
    X_improved, y, 
    test_size=0.2, 
    random_state=42
)

# Fit improved model
model_improved = LinearRegression()
model_improved.fit(X_train_imp, y_train_imp)

# Evaluate improved model
y_test_pred_imp = model_improved.predict(X_test_imp)
r2_test_imp = r2_score(y_test_imp, y_test_pred_imp)
mae_test_imp = mean_absolute_error(y_test_imp, y_test_pred_imp)
rmse_test_imp = np.sqrt(mean_squared_error(y_test_imp, y_test_pred_imp))

# Compare before vs after
print("\n" + "="*70)
print("RESULTS COMPARISON")
print("="*70)
print(f"\n{'Metric':<25} {'Original':<15} {'Improved':<15} {'Change':<15}")
print("-"*70)
print(f"{'Test R²':<25} {r2_test:<15.4f} {r2_test_imp:<15.4f} {r2_test_imp - r2_test:+.4f}")
print(f"{'Test MAE (rides)':<25} {mae_test:<15.2f} {mae_test_imp:<15.2f} {mae_test_imp - mae_test:+.2f}")
print(f"{'Test RMSE (rides)':<25} {rmse_test:<15.2f} {rmse_test_imp:<15.2f} {rmse_test_imp - rmse_test:+.2f}")

print("\n" + "="*70)
if r2_test_imp > r2_test:
    print("✓ IMPROVEMENT: Test R² increased!")
    print(f"  The model now explains {(r2_test_imp - r2_test)*100:.2f}% more variance.")
else:
    print("✗ NO IMPROVEMENT: Test R² decreased or stayed the same.")
    print("  This feature change did not help predictive performance.")
print("="*70)

# Show coefficient for new feature
if 'temp_f_squared' in X_improved.columns:
    new_feature_idx = list(X_improved.columns).index('temp_f_squared')
    temp_squared_coef = model_improved.coef_[new_feature_idx]
    print(f"\nCoefficient for temp_f_squared: {temp_squared_coef:.6f}")
    
print("\nNote: Even negative results are valuable! They tell us what doesn't work.")